# Model Comparison Dashboard

This notebook loads saved metrics and artifacts for the active Stage B models and produces presentation-ready comparison plots.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import config

OUT = Path(config.OUTPUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (8,4)
plt.rcParams["axes.grid"] = True
sns.set_context("talk")

## 1. Load metrics

In [ ]:

metric_files = {
    "pooled_factor_pvar": OUT / "vmf_pvar_pooled_metrics.csv",
    "switching_pvar": OUT / "switching_pvar_k3_metrics.csv",
    "switching_factor_pvar": OUT / "switching_factor_pvar_k3_metrics.csv",
    "unit_switching_pvar": OUT / "unit_switching_pvar_k3_metrics.csv",
    "slds_panel": OUT / "slds_panel_k3_r3_metrics.csv",
    "baselines": OUT / "vmf_baseline_comparison.csv",
}

loaded = {}
for name, path in metric_files.items():
    if path.exists():
        loaded[name] = pd.read_csv(path)
    else:
        print(f"Missing: {path}")

loaded.keys()


In [ ]:

frames = []
for name, df in loaded.items():
    if name == "baselines":
        frames.append(df.copy())
    else:
        tmp = df.copy()
        if "mode" not in tmp.columns:
            tmp["mode"] = name
        frames.append(tmp)

metrics_df = pd.concat(frames, ignore_index=True, sort=False)
metrics_df


## 2. Main metrics table

In [ ]:

cols = [c for c in ["mode", "mse", "rmse", "r2", "accuracy", "kl", "cross_entropy"] if c in metrics_df.columns]
summary = metrics_df[cols].copy()
summary = summary.sort_values("r2", ascending=False)
summary


## 3. Bar charts for model comparison

In [ ]:

plot_cols = [c for c in ["r2", "rmse", "accuracy", "kl"] if c in summary.columns]
for c in plot_cols:
    plt.figure(figsize=(10,4))
    order = summary.sort_values(c, ascending=(c in ["rmse", "kl"]))
    plt.bar(order["mode"], order[c])
    plt.xticks(rotation=35, ha="right")
    plt.title(f"Model comparison: {c}")
    plt.tight_layout()
    plt.show()


## 4. Load Option B artifacts

In [ ]:

art_b_path = OUT / "unit_switching_pvar_k3_artifacts.npz"
assert art_b_path.exists(), f"Missing {art_b_path}"
art_b = np.load(art_b_path, allow_pickle=True)
art_b.files


In [ ]:

gamma = art_b["regime_prob_filt"]   # (N, T_oof, K)
A = art_b["A"]
B = art_b["B"]
Pi = art_b["Pi"]
feature_names = art_b["feature_names"]
y_true = art_b["y_true_oof"]
y_pred = art_b["y_pred_oof"]

gamma.shape, A.shape, B.shape, Pi.shape


## 5. Option B regime usage shares

In [ ]:

dom_regime = np.argmax(gamma, axis=2)
shares = np.bincount(dom_regime.reshape(-1), minlength=gamma.shape[2])
shares = shares / shares.sum()
share_df = pd.DataFrame({
    "regime": [f"Regime {k+1}" for k in range(gamma.shape[2])],
    "share": shares,
})
share_df


In [ ]:

plt.figure(figsize=(6,4))
plt.bar(share_df["regime"], share_df["share"])
plt.title("Option B: regime usage shares")
plt.ylabel("Fraction of unit-time OOS observations")
plt.tight_layout()
plt.show()


## 6. Option B average regime probability over time

In [ ]:

mean_gamma = gamma.mean(axis=0)
plt.figure(figsize=(10,4))
for k in range(mean_gamma.shape[1]):
    plt.plot(mean_gamma[:, k], label=f"Regime {k+1}")
plt.legend()
plt.title("Option B: average regime probability across units")
plt.xlabel("OOS time index")
plt.ylabel("Probability")
plt.tight_layout()
plt.show()


## 7. Option B dominant regime heatmap

In [ ]:

plt.figure(figsize=(12,6))
plt.imshow(dom_regime, aspect="auto", interpolation="nearest")
plt.title("Option B: dominant regime by unit and OOS time")
plt.xlabel("OOS time index")
plt.ylabel("Unit")
plt.colorbar()
plt.tight_layout()
plt.show()


## 8. Option B AR matrix summaries

In [ ]:

rows = []
for k in range(A.shape[0]):
    Ak = A[k]
    diag_mean = np.mean(np.diag(Ak))
    offdiag = Ak - np.diag(np.diag(Ak))
    offdiag_mean = np.mean(np.abs(offdiag))
    rows.append({
        "regime": f"Regime {k+1}",
        "mean_diag_A": diag_mean,
        "mean_abs_offdiag_A": offdiag_mean,
    })
A_summary = pd.DataFrame(rows)
A_summary


In [ ]:

fig, axes = plt.subplots(1, A.shape[0], figsize=(5*A.shape[0], 4), constrained_layout=True)
if A.shape[0] == 1:
    axes = [axes]
for k in range(A.shape[0]):
    im = axes[k].imshow(A[k], aspect="auto")
    axes[k].set_title(f"Regime {k+1}: AR matrix")
    axes[k].set_xlabel("Response state")
    axes[k].set_ylabel("Lagged state")
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()


## 9. Option B top covariates by regime

In [ ]:

for k in range(B.shape[0]):
    importance = np.mean(np.abs(B[k]), axis=1)
    df = pd.DataFrame({
        "covariate": feature_names,
        "importance": importance,
    }).sort_values("importance", ascending=False)
    print(f"\nTop 15 covariates in Regime {k+1}")
    display(df.head(15))


## 10. Option B confusion matrix

In [ ]:

true_state = np.argmax(y_true, axis=1)
pred_state = np.argmax(y_pred, axis=1)
conf = pd.crosstab(pd.Series(true_state, name="true"), pd.Series(pred_state, name="pred"))
conf_row = conf.div(conf.sum(axis=1), axis=0)
conf_row


In [ ]:

plt.figure(figsize=(6,5))
sns.heatmap(conf_row, annot=True, fmt=".2f", cmap="viridis")
plt.title("Option B: row-normalized confusion matrix")
plt.tight_layout()
plt.show()


## 11. Export-ready figure suggestions

- model comparison bars
- Option B regime shares
- Option B dominant regime heatmap
- Option B AR matrices
- Option B confusion matrix